# Asistente RAG UTB

Chatbot universitario para consulta de reglamentos.

En el fichero `data/preguntas/preguntas.csv` se encuentran las preguntas que se usarán para evaluar la línea base contra RAG. El CSV está clasificado por tipo de pregunta (articulo_especifico, interpretacion_simple, etc.) y por documento (RE2026, CJ2025, FOTO, etc.). 

## Instalación de librerías

In [ ]:
%pip install google-genai pymupdf numpy pandas langchain langchain_text_splitters

## Clave de Gemini API

In [1]:
import os
from getpass import getpass

# Ejecutar si no se ha establecido la variable GEMINI_API_KEY con la api key
GEMINI_API_KEY = getpass("Ingrese la api key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

if os.getenv("GEMINI_API_KEY") == GEMINI_API_KEY:
    print("Variable de entorno GEMINI_API_KEY establecida.")

Variable de entorno GEMINI_API_KEY establecida.


## Añadir ruta raíz del proyecto al PATH de Python

Necesario para importar los módulos

In [1]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## Etapa 1. Fetching de PDFs institucionales

In [3]:
from time import sleep
import requests

from config import (
    RUTAS_ARCHIVOS_PDF,
    URLS
)

# Descargar PDFs y guardar en data/pdf

for url, ruta in zip(URLS, RUTAS_ARCHIVOS_PDF):
    resp = requests.get(url)
    with open(ruta, "wb") as f:
        f.write(resp.content)

    print(f"Documento guardado: {ruta}")
    
    sleep(1)

Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/Reglamento-Ruta-de-autonomia.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/reglamento-del-CJ-26-09-25.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/Reglamento-Estudiantil-2026.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/REGLAMENTO-DE-USO-DE-LOS-LABORATORIOS-DE-PSICOLOGIA.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/reglamento_estudio_de_television_y_laboratorio_de_posproduccion_0_0.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/reglamento_laboratorio_de_fotografia_0.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/pdf/reglamento_laboratorio_de_sonido_0.pdf
Documento guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb

## Etapa 2. Extracción y preprocesamiento de texto

In [4]:
import pymupdf

from config import (
    RUTAS_ARCHIVOS_PDF,
    RUTAS_ARCHIVOS_TXT
)

reglamentos = []       # textos de los pdfs

# -------------- Reglamento de autonomía y del consultorio jurídico ------ #

for ruta in RUTAS_ARCHIVOS_PDF[0:2]:
        
    reglamento = ""   # aqui irá el texto completo del reglamento actual
    doc = pymupdf.open(ruta)  # leer el pdf

    for i, pagina in enumerate(doc):    # iterar por paginas
        texto = pagina.get_text().strip()

        reemplazo = "REGLAMENTO DE LA RUTA DE AUTONOMÍA Y ÉXITO PROFESIONALPARA ESTUDIANTES DE PREGRADO PRESENCIAL\n" + str(i+1) if ((i+1) % 2 != 0) else ""
        reemplazo = "UNIVERSIDAD TECNOLÓGICA DE BOLÍVAR\n" + str(i+1) if ((i+1) % 2 == 0) else reemplazo
        texto = texto.replace(reemplazo, "")  # quitar texto que se repite en todas las paginas
        texto = texto.replace("CAPITULO III", "CAPÍTULO III")  # corregir una I sin tilde
        texto += "\n"
        reglamento += texto
    
    reglamentos.append(reglamento)

# --------------Reglamento estudiantil ------------------------------- #

reglamento = ""
ruta = RUTAS_ARCHIVOS_PDF[2]
doc = pymupdf.open(ruta)
reemplazo = (
    "Reglamento Estudiantil\n"
    "Aprobado en Acta No. 03 del 17 de mayo de 2024 del Consejo Académico Resolución \n"
    "No. 01 de 2024 y Modificado por la Resolución del Consejo Académico No. 01 de 2026"
)

for i, pagina in enumerate(doc):
    if (i+1) in [2,3,4]:   # omitir paginas de CONTENIDO
        continue
    texto = pagina.get_text().strip()
    impar = str(i+1) + '\n' if ((i+1) % 2 != 0) else ""
    par = '\n' + str(i+1) if ((i+1) % 2 == 0) else ""
    texto = texto.replace(f"{impar}{reemplazo}{par}", "")  # quitar texto que se repite en todas las paginas
    texto += "\n"
    reglamento += texto

reglamentos.append(reglamento)

# ------------ Otros reglamentos ------------------------------------- #

for ruta in RUTAS_ARCHIVOS_PDF[3:]:
    reglamento = ""
    doc = pymupdf.open(ruta)

    for i, pagina in enumerate(doc):
        texto = pagina.get_text().strip()
        texto = texto.replace("\n", "-- ")  # Reemplazar viñeta
        texto = texto.replace("", "-- ") # Reemplazar viñeta
        texto += "\n"
        reglamento += texto
    
    reglamentos.append(reglamento)

# -------------- Guardar en archivos txt --------------------------- #

for reglamento, ruta in zip(reglamentos, RUTAS_ARCHIVOS_TXT):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(reglamento)

    print(f"Texto guardado: {ruta}")

Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/Reglamento-Ruta-de-autonomia.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/reglamento-del-CJ-26-09-25.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/Reglamento-Estudiantil-2026.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/REGLAMENTO-DE-USO-DE-LOS-LABORATORIOS-DE-PSICOLOGIA.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/reglamento_estudio_de_television_y_laboratorio_de_posproduccion_0_0.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/reglamento_laboratorio_de_fotografia_0.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/reglamento_laboratorio_de_sonido_0.txt
Texto guardado: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/texto/reglam

## Etapa 3. Línea base: LLM sin RAG

### Generación de respuestas

In [2]:
from src.ejecutar_experimento import ejecutar_experimento
from config import (
    PLANTILLA_PROMPT_BASELINE,
    RUTA_RESPUESTAS_BASELINE
)
import os
import sys

def construir_prompt(pregunta):
    return PLANTILLA_PROMPT_BASELINE.format(
        pregunta=pregunta.strip(),
    ), {}

def linea_base(*, ver_existentes=False, reemplazar=False):
    try:
        ejecutar_experimento.main(ruta_salida=RUTA_RESPUESTAS_BASELINE,
                                  construir_prompt=construir_prompt,
                                  ver_existentes=ver_existentes,
                                  reemplazar=reemplazar)
    except KeyboardInterrupt:
        print("\nInterrumpido por el usuario. Las respuestas guardadas se conservan.")
        sys.exit(130)

In [8]:
if os.path.exists(RUTA_RESPUESTAS_BASELINE):
    opcion = input(
        f"Ya existe la línea base en {RUTA_RESPUESTAS_BASELINE}.\n"
        "¿Quieres ver las respuestas ya generadas (v), regenerarlas (r) "
        "o continuar generando las respuestas (c)? [v/r/c]: "
    ).strip().lower()

    if opcion == "v":
        linea_base(ver_existentes=True)
    elif opcion == "r":
        linea_base(reemplazar=True)
    elif opcion == "c":
        linea_base()
else:
    linea_base()

Modelo: gemini-3.5-flash-lite
Preguntas: 50
Salida: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/linea_base/respuestas_baseline.csv
----------------------------------------------------------------------
[1/50] Q01 [articulo_especifico] ¿Cuál es la nota mínima aprobatoria de un curso de pregrado y de un curso de posgrado en l...
    OK (1.6s, 99 palabras)
[2/50] Q02 [articulo_especifico] ¿Cuántas semanas dura un periodo académico regular de pregrado en la modalidad presencial?...
    OK (0.91s, 76 palabras)
[3/50] Q03 [articulo_especifico] ¿Cuántos créditos como máximo puede matricular un estudiante de pregrado presencial en mat...
    OK (1.23s, 146 palabras)
[4/50] Q04 [articulo_especifico] ¿A cuántas horas de trabajo del estudiante equivale un crédito académico en la UTB?...
    OK (1.03s, 138 palabras)
[5/50] Q05 [articulo_especifico] ¿Cuál es el Promedio Ponderado Acumulado (PPA) mínimo que debe mantener un estudiante de p...
    OK (1.14s, 108 palabras)
[

In [3]:
# Ver respuestas generadas

linea_base(ver_existentes=True)

RESPUESTAS YA GENERADAS: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/linea_base/respuestas_baseline.csv
▪ ID: Q01
▪ Pregunta: ¿Cuál es la nota mínima aprobatoria de un curso de pregrado y de un curso de posgrado en la UTB?
▪ Respuesta Esperada: En pregrado la nota mínima aprobatoria es 3,0; en posgrado es 3,5. La práctica profesional de pregrado también exige 3,5.
▪ Respuesta Generada: ¡Hola! Como asistente académico de la UTB, con gusto resuelvo tu duda.

En la Universidad Tecnológica de Bolívar (UTB), las notas se califican en una escala de **0.0 a 5.0**. 

Las notas mínimas aprobatorias son las siguientes:

*   **Pregrado:** La nota mínima para aprobar un curso es **3.0**.
*   **Posgrado:** La nota mínima para aprobar un curso de especialización o maestría es **3.5**.

Te sugiero revisar el Reglamento Estudiantil vigente en la página web de la universidad para conocer más detalles sobre evaluaciones y requisitos de permanencia. ¡Mucho éxito en tus estudios

### Calificación

En este punto se debe editar el archivo `experiments/linea_base/respuestas_baseline.csv` con algún programa externo (Excel, LibreOffice Calc) para asignar una calificación a cada respuesta en la columna correspondiente. **En la línea base la fidelidad no aplica pues no hay recuperación de documentos, por lo que debe ser calificada con 0.**

La escala de calificación es:
- **Exactitud (0-3):** 0 = incorrecta; 1 = correcta en algo pero con errores importantes; 2 = parcialmente correcta u omite datos clave; 3 = correcta y completa frente a la respuesta esperada.
- ~~**Fidelidad (0-3):** 0 = no se sustenta en las fuentes o las contradice; 1 = mezcla datos fieles con inventados; 2 = mayormente fiel con imprecisiones; 3 = todos los datos relevantes son verificables en las fuentes.~~
- **Alucinación (0-2):** 0 = no inventa; 1 = inventa un detalle menor (cifra, plazo, nombre, correo); 2 = inventa hechos centrales o contradice la fuente.

Después, ejecuta la siguiente celda.

In [3]:
from src.ejecutar_experimento import calcular_metricas
from config import (
    RUTA_RESPUESTAS_BASELINE,
    DIR_RESULTADOS_BASELINE
)

while calcular_metricas.main(
    RUTA_RESPUESTAS_BASELINE,
    DIR_RESULTADOS_BASELINE,
    "Métricas de la línea base (sin RAG)",
) == 1:
    print("\nTermina la calificación y presiona ENTER para continuar...")
    input()

Archivo: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/linea_base/respuestas_baseline.csv
Filas: 50 | Calificadas: 50 | Sin calificar: 0
Errores de API: 0 | Sin respuesta generada: 0

MÉTRICAS PRINCIPALES
  Exactitud (%): 39.33
  Fidelidad (%): N/A
  Tasa de alucinación (%): 82.0
  Respuesta perfecta (%): N/A

OTRAS MÉTRICAS
  Preguntas totales: 50
  Preguntas calificadas: 50
  Cobertura de calificación (%): 100.0
  Errores de API: 0
  Sin respuesta generada: 0
  Exactitud plena (=3) (%): 18.0
  Exactitud aceptable (>=2) (%): 40.0
  Exactitud baja (<=1) (%): 60.0
  Fidelidad plena (%): N/A
  Fidelidad baja (<=1) (%): N/A
  Sin alucinación (%): 18.0
  Alucinación grave (=2) (%): 48.0

MÉTRICAS POR TIPO
                      tipo  n_evaluadas  %exactitud  tasa_alucinacion
       articulo_especifico           10   50.000000         60.000000
     interpretacion_simple            8   50.000000         87.500000
            multi_articulo            8   58.333333   

## Etapa 4. Procesamiento de texto: división en chunks o fragmentos

In [6]:
from src.chunking.chunking import (
    MAXIMO,
    num_palabras,
    extraer,
    insertar_chunks_en_db
)

from config import (
    CARPETA_CHUNKS,
    RUTAS_ARCHIVOS_TXT
)


import pandas as pd
import os
import re

reglamentos = []       # textos
titulos_capitulos = [] # titulos de los capitulos de los reglamentos
patrones = []          # patrones para encontrar los chunks o articulos
nombres = []           # nombres de los reglamentos


# --------- Reglamento ruta de autonomia y consultorio juridico ------- #

for ruta in RUTAS_ARCHIVOS_TXT[:2]:
    with open(ruta, encoding="utf8") as f:
        reglamento = f.read()

    patron_cap = re.compile(r"(?s)(CAP[ÍI]TULO\s*.*?)\s*Art[íi]culo")
    patron_art = re.compile(r"Artículo .*?\.")

    reglamentos.append(reglamento)
    titulos_capitulos.append( patron_cap.findall(reglamento) )
    patrones.append(patron_art)

nombres.append("Reglamento Ruta de Autonomía")
nombres.append("Reglamento del Consultorio Jurídico")

# --------------Reglamento estudiantil ------------------------------- #
ruta = RUTAS_ARCHIVOS_TXT[2]
with open(ruta, encoding="utf8") as f:
    reglamento = f.read()

patron_cap = re.compile(r"(?m)^[IVXLM]+\.(?!\s+(?:LENGUAS EXTRANJERAS|RECLAMOS))\s[\s\S]*?(?=\nARTÍCULO|\nA\.)")
patron_art = re.compile(r"ARTÍCULO .*?:")

titulos_capitulos.append( ["PRESENTACIÓN", "GLOSARIO"] + patron_cap.findall(reglamento) )
patrones.append(patron_art)
nombres.append("Reglamento Estudiantil")
reglamentos.append(reglamento)


# ------------ Otros reglamentos ------------------------------------- #

cap_de_otros_reglamentos = [
    ["REGLAMENTO DE USO DE LOS LABORATORIOS DE PSICOLOGÍA \nLABPSIC"],
    
    ["REGLAMENTO ESTUDIO DE TELEVISIÓN Y LABORATORIO DE\nPOSPRODUCCIÓN.",
    "DISPOSICIONES GENERALES.", "Utilización de equipos al interior y fuera de la Universidad\nTecnológica de Bolívar.",
    "Son causales para perder el turno de prácticas\ny uso de equipos."],
    
    ["REGLAMENTO LABORATORIO DE FOTOGRAFÍA.", "DISPOSICIONES GENERALES.", 
    "Utilización de equipos al interior de la Universidad\nTecnológica de Bolívar",
    "Utilización de equipos fuera de la Universidad Tecnológica de\nBolívar.",
    "Son causales para perder el turno de\nprácticas."],
    
    ["REGLAMENTO LABORATORIO DE SONIDO", "DISPOSICIONES GENERALES.", "Son causales para perder el turno de\nprácticas."],
    
    ["REGLAMENTO SALA DE REDACCIÓN.", "DISPOSICIONES GENERALES."]
]

for ruta, titulos_cap in zip(RUTAS_ARCHIVOS_TXT[3:], cap_de_otros_reglamentos):
    with open(ruta, encoding="utf8") as f:
        reglamentos.append(f.read())

    if str(ruta).count("PSICOLOGIA") > 0:  # Si es el reglamento de laboratorios de psicologia,
        patron_art = re.compile(r"\d+\.")  # insertar una expresion regular diferente
    else:
        patron_art = re.compile("-- ")

    titulos_capitulos.append(titulos_cap)
    patrones.append(patron_art)

nombres.append("Reglamento de uso de los laboratorios de psicología")
nombres.append("Reglamento del estudio de televisión y laboratorio de posproducción")
nombres.append("Reglamento del laboratorio de fotografía")
nombres.append("Reglamento del laboratorio de sonido")
nombres.append("Reglamento sala de redacción")

# ------------------------ Crear dataframe ------------------ #

df = pd.DataFrame({
    "chunk": [],
    "capitulo": [], 
    "documento": [],
    "embeddings": []
})

for i in range(len(reglamentos)):
    df = insertar_chunks_en_db(reglamentos[i], nombres[i], titulos_capitulos[i], df, patrones[i])

df["n_palabras"] = (
    df["chunk"].str.split().str.len()
)

print("\n- Chunks que exceden las 500 palabras: ", len(df[df["n_palabras"] > 500]))
display(df[df["n_palabras"] > 500])
print()

print("\n- Chunks con menos de 10 palabras:", len(df[ df["n_palabras"] < 10 ]), ". Se eliminarán:\n")
display(df[ df["n_palabras"] < 10 ])
print()

# Eliminar registros de menos de 10 palabras
df = df[ df["n_palabras"] >= 10 ].reset_index(drop=True)

print("\n- Estadisticas de longitud de chunks:\n")
display(df["n_palabras"].describe())

print("\n- Dataframe resultante:\n")
display(df)

print(f"{len(df)} registros agregados.")


# -------------- Guardar en csv y html ------------------------ #

if not os.path.isdir(CARPETA_CHUNKS):
    os.mkdir(CARPETA_CHUNKS)

df.to_csv(CARPETA_CHUNKS / "db.csv", index=False)
df.to_html(CARPETA_CHUNKS / "db.html")

print(f"Guardado en {CARPETA_CHUNKS}/db.csv")
print(f"Guardado en {CARPETA_CHUNKS}/db.html")

df_chunks = df


- Chunks que exceden las 500 palabras:  2


,chunk,capitulo,documento,embeddings,n_palabras
12,Artículo 3. Principios. El Consultorio Jurídic...,CAPÍTULO I Disposiciones generales,Reglamento del Consultorio Jurídico,NaN,546
19,Artículo 14. Deberes de los estudiantes de Con...,CAPÍTULO IV Estudiantes de Consultorio Jurídi...,Reglamento del Consultorio Jurídico,NaN,999




- Chunks con menos de 10 palabras: 4 . Se eliminarán:



,chunk,capitulo,documento,embeddings,n_palabras
64,Centro de Medios\nPrograma de Comunicación Social,INFORMACIÓN PREVIA,Reglamento del estudio de televisión y laborat...,NaN,7
70,Centro de Medios \nPrograma de Comunicación So...,INFORMACIÓN PREVIA,Reglamento del laboratorio de fotografía,NaN,7
77,Centro de Medios \nPrograma de Comunicación So...,INFORMACIÓN PREVIA,Reglamento del laboratorio de sonido,NaN,7
81,Centro de Medios Alcatraz.\nPrograma de Comuni...,INFORMACIÓN PREVIA,Reglamento sala de redacción,NaN,8




- Estadisticas de longitud de chunks:



count     81.000000
mean     287.592593
std      177.279284
min       44.000000
25%      136.000000
50%      301.000000
75%      442.000000
max      999.000000
Name: n_palabras, dtype: float64


- Dataframe resultante:



,chunk,capitulo,documento,embeddings,n_palabras
0,Reglamento de la \nRuta de autonomía y \néxito...,INFORMACIÓN PREVIA,Reglamento Ruta de Autonomía,NaN,173
1,Artículo 1º.- Objeto.\nEl presente reglamento ...,CAPÍTULO I DISPOSICIONES GENERALES,Reglamento Ruta de Autonomía,NaN,242
2,Artículo 5º.- Para efectos del presente reglam...,CAPÍTULO II DEFINICIONES Y CRITERIOS DE INTERP...,Reglamento Ruta de Autonomía,NaN,167
3,Artículo 6º.- Habilitadores de éxito.\nLa Ruta...,CAPÍTULO III ESTRUCTURA DE LA RUTA Y SISTEMA D...,Reglamento Ruta de Autonomía,NaN,153
4,Artículo 9º.- Insignia Básica Core Skills.\nLa...,CAPÍTULO IV HABILITADOR I: HABILIDADES PARA EL...,Reglamento Ruta de Autonomía,NaN,80
...,...,...,...,...,...
76,Es responsabilidad del Centro de Medios Alcatr...,DISPOSICIONES GENERALES.,Reglamento del laboratorio de sonido,NaN,489
77,-- Presentarse 10 minutos después del horario ...,Son causales para perder el turno de prácticas.,Reglamento del laboratorio de sonido,NaN,187
78,El presente reglamento tiene como objetivo est...,REGLAMENTO SALA DE REDACCIÓN.,Reglamento sala de redacción,NaN,55
79,Es responsabilidad del Centro de Medios Alcatr...,DISPOSICIONES GENERALES.,Reglamento sala de redacción,NaN,497


81 registros agregados.
Guardado en /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/chunks/db.csv
Guardado en /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/chunks/db.html


## Etapa 5. Embeddings

In [ ]:
import pandas as pd
from config import CARPETA_CHUNKS
# Leer csv de chunks si no se ejecutó la etapa anterior
ruta_chunks = CARPETA_CHUNKS / "db.csv"
if not ruta_chunks.exists():
    print("No existe la base de chunks. Ejecuta primero 'preprocesamiento' y 'procesamiento'.")
df_chunks = pd.read_csv(ruta_chunks)

In [9]:
from config import crear_cliente
import time

from src.embedding.embedding import (
    RUTA_CSV,
    RUTA_HTML,
    CARACTERES_POR_TOKEN,
    LIMITE_TOKENS_POR_MINUTO,
    ESPERA_ENTRE_SOLICITUDES,
    convertir_embedding_guardado,
    cargar_progreso,
    obtener_segundos_reintento,
    guardar_progreso,
    crear_embedding_con_reintentos
)

# Conectar a Gemini API

client = crear_cliente()
df_chunks = cargar_progreso(df_chunks)
cantidad = len(df_chunks)

# Lleva una ventana aproximada de consumo de tokens por 60 segundos.
eventos_tokens = []
creado = 0

for i, fila in df_chunks.iterrows():
    embedding_actual = convertir_embedding_guardado(fila.get("embeddings"))
    if embedding_actual:
        continue

    ahora = time.monotonic()
    eventos_tokens = [(momento, tokens) for momento, tokens in eventos_tokens if ahora - momento < 60]
    tokens_estimados = max(1, len(str(fila["chunk"])) // CARACTERES_POR_TOKEN)
    tokens_en_minuto = sum(tokens for _, tokens in eventos_tokens)

    if tokens_en_minuto + tokens_estimados > LIMITE_TOKENS_POR_MINUTO:
        espera = 60 - (ahora - eventos_tokens[0][0])
        print(f"Límite aproximado de 30000 tokens por minuto. Esperando {espera:.1f} segundos.")
        time.sleep(max(1.0, espera + 1.0))
        eventos_tokens = []

    # Llamar al modelo de embeddings. Se le pasa informacion sobre el documento,
    # capitulo y el chunk. Asi el modelo de embeddings captura mejor el contexto
    # de cada fragmento
    texto = f"{fila['documento']}, {fila['capitulo']}\n\n{fila['chunk']}"
    embedding = crear_embedding_con_reintentos(client, texto)

    momento_envio = time.monotonic()
    eventos_tokens.append((momento_envio, tokens_estimados))
    df_chunks.at[i, "embeddings"] = embedding
    creado += 1

    # Persiste inmediatamente cada fila, no solo al terminar todo el proceso.
    guardar_progreso(df_chunks)

    if creado % 5 == 0:
        completados = sum(
            convertir_embedding_guardado(df_chunks.at[j, "embeddings"]) is not None
            for j in range(cantidad)
        )
        print(f"Progreso: {completados} embeddings de {cantidad}")

    time.sleep(ESPERA_ENTRE_SOLICITUDES)

completados = sum(
    convertir_embedding_guardado(df_chunks.at[j, "embeddings"]) is not None
    for j in range(cantidad)
)
print(f"Progreso: completados {completados} embeddings")

if completados:
    primer_embedding = next(
        valor for valor in df_chunks["embeddings"]
        if convertir_embedding_guardado(valor)
    )
    primer_embedding = convertir_embedding_guardado(primer_embedding)
    print("Tamaño de vector:", len(primer_embedding))

guardar_progreso(df_chunks)

print(f"Guardado en {RUTA_CSV}")
print(f"Guardado en {RUTA_HTML}")

df = df_chunks

Progreso: 5 embeddings de 81
Progreso: 10 embeddings de 81
Progreso: 15 embeddings de 81
Progreso: 20 embeddings de 81
Progreso: 25 embeddings de 81
Progreso: 30 embeddings de 81
Progreso: 35 embeddings de 81
Progreso: 40 embeddings de 81
Progreso: 45 embeddings de 81
Progreso: 50 embeddings de 81
Progreso: 55 embeddings de 81
Progreso: 60 embeddings de 81
Progreso: 65 embeddings de 81
Progreso: 70 embeddings de 81
Progreso: 75 embeddings de 81
Progreso: 80 embeddings de 81
Progreso: completados 81 embeddings
Tamaño de vector: 3072
Guardado en /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/embeddings/db.csv
Guardado en /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/data/embeddings/db.html


## Etapa 6. Línea RAG: LLM + RAG

### Generación de respuestas

In [6]:
import pandas as pd
from config import (
    CARPETA_EMBEDDINGS
)
import ast

# Leer embeddings del csv si no se ejecutó la etapa anterior

ruta_embeddings = CARPETA_EMBEDDINGS / "db.csv"
if not ruta_embeddings.exists():
    print("No existe la base de embeddings. Ejecuta primero 'procesamiento' y 'embedding'.")
df = pd.read_csv(ruta_embeddings, converters={"embeddings": ast.literal_eval})

In [7]:
import sys
from src.ejecutar_experimento import ejecutar_experimento
import numpy as np

from config import (
    PLANTILLA_PROMPT_RAG,
    RUTA_RESPUESTAS_RAG,
    crear_cliente,
    crear_embedding
)

def find_top_k_passages(client, query, df_embeddings, k=5):
    query_embedding = crear_embedding(client, query)

    scores = np.dot(
        np.stack(df_embeddings["embeddings"]),
        query_embedding
    )

    top_k_idx = np.argsort(scores)[-k:][::-1]

    results = df_embeddings.iloc[top_k_idx].copy()
    results["score"] = scores[top_k_idx]

    return results


def preparar_texto(results):
    lista_docs = []
    for fila in results.iloc:
        texto = f"\n - DOCUMENTO: {fila["documento"]}"
        texto += f"\n - CAPITULO: {fila["capitulo"].replace("\n", " ")}"
        texto += f"\n - SCORE: {fila["score"]}"
        texto += f"\n - TEXTO: «{fila["chunk"].strip()}»"
        texto += "\n\n--------------------------------------------------\n"
        lista_docs.append(texto)

    return lista_docs

def crear_funcion_prompt_rag(df_embeddings, k=5):
    cliente = crear_cliente()

    def construir_prompt(pregunta):
        results = find_top_k_passages(
            cliente,
            pregunta.strip(),
            df_embeddings,
            k
        )
        lista_docs = preparar_texto(results)
        docs = "".join(lista_docs)

        prompt = PLANTILLA_PROMPT_RAG.format(
            pregunta=pregunta.strip(),
            docs=docs,
        )

        metadatos = {}
        for i in range(len(lista_docs)):
            metadatos[f"doc_enviado_{i}"] = lista_docs[i]
        
        return prompt, metadatos

    return construir_prompt


def linea_rag(df_embeddings, *, ver_existentes=False, reemplazar=False):

    construir_prompt = crear_funcion_prompt_rag(df_embeddings)

    try:
        ejecutar_experimento.main(
            ruta_salida=RUTA_RESPUESTAS_RAG,
            construir_prompt=construir_prompt,
            ver_existentes=ver_existentes,
            reemplazar=reemplazar,
        )
    except KeyboardInterrupt:
        print("\nInterrumpido por el usuario. Las respuestas guardadas se conservan.")
        sys.exit(130)

In [12]:
if os.path.exists(RUTA_RESPUESTAS_RAG):
    opcion = input(
        f"Ya existe la línea RAG en {RUTA_RESPUESTAS_RAG}.\n"
        "¿Quieres ver las respuestas ya generadas (v), regenerarlas (r) "
        "o continuar generando las respuestas (c)? [v/r/c]: "
    ).strip().lower()

    if opcion == "v":
        linea_rag(df, ver_existentes=True)
    elif opcion == "r":
        linea_rag(df, reemplazar=True)
    elif opcion == "c":
        linea_rag(df)
else:
    linea_rag(df)

Modelo: gemini-3.5-flash-lite
Preguntas: 50
Salida: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/rag/respuestas_rag.csv
----------------------------------------------------------------------
[1/50] Q01 [articulo_especifico] ¿Cuál es la nota mínima aprobatoria de un curso de pregrado y de un curso de posgrado en l...
    OK (1.02s, 29 palabras)
[2/50] Q02 [articulo_especifico] ¿Cuántas semanas dura un periodo académico regular de pregrado en la modalidad presencial?...
    OK (0.9s, 22 palabras)
[3/50] Q03 [articulo_especifico] ¿Cuántos créditos como máximo puede matricular un estudiante de pregrado presencial en mat...
    OK (1.09s, 79 palabras)
[4/50] Q04 [articulo_especifico] ¿A cuántas horas de trabajo del estudiante equivale un crédito académico en la UTB?...
    OK (1.1s, 47 palabras)
[5/50] Q05 [articulo_especifico] ¿Cuál es el Promedio Ponderado Acumulado (PPA) mínimo que debe mantener un estudiante de p...
    OK (1.0s, 47 palabras)
[6/50] Q06 [articu

In [8]:
linea_rag(df, ver_existentes=True)

RESPUESTAS YA GENERADAS: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/rag/respuestas_rag.csv
▪ ID: Q01
▪ Pregunta: ¿Cuál es la nota mínima aprobatoria de un curso de pregrado y de un curso de posgrado en la UTB?
▪ Respuesta Esperada: En pregrado la nota mínima aprobatoria es 3,0; en posgrado es 3,5. La práctica profesional de pregrado también exige 3,5.
▪ Respuesta Generada: Según el Reglamento Estudiantil de la UTB (Artículo 51), las notas mínimas aprobatorias son:

*   **Cursos de pregrado:** 3,0
*   **Cursos de posgrado:** 3,5
▪ Documento: RE2026	▪ Fuente: Reglamento Estudiantil UTB 2026, Art. 51 y Parágrafo 1
▪ Exactitud calificada: 3	▪ Fidelidad calificada: 3	▪ Alucinacion calificada: 0
--------------------------------------------------------------------------------
▪ ID: Q02
▪ Pregunta: ¿Cuántas semanas dura un periodo académico regular de pregrado en la modalidad presencial?
▪ Respuesta Esperada: 16 semanas.
▪ Respuesta Generada: Según el Reglamento Est

### Calificación

En este punto se debe editar el archivo `experiments/rag/respuestas_rag.csv` con algún programa externo (Excel, LibreOffice Calc) para asignar una calificación a cada respuesta en la columna correspondiente.

La escala de calificación es:
- **Exactitud (0-3):** 0 = incorrecta; 1 = correcta en algo pero con errores importantes; 2 = parcialmente correcta u omite datos clave; 3 = correcta y completa frente a la respuesta esperada.
- **Fidelidad (0-3):** 0 = no se sustenta en las fuentes o las contradice; 1 = mezcla datos fieles con inventados; 2 = mayormente fiel con imprecisiones; 3 = todos los datos relevantes son verificables en las fuentes.
- **Alucinación (0-2):** 0 = no inventa; 1 = inventa un detalle menor (cifra, plazo, nombre, correo); 2 = inventa hechos centrales o contradice la fuente.

Después, ejecuta la siguiente celda.

In [16]:
from src.ejecutar_experimento import calcular_metricas
from config import (
    RUTA_RESPUESTAS_RAG,
    DIR_RESULTADOS_RAG
)

while calcular_metricas.main(
    RUTA_RESPUESTAS_RAG,
    DIR_RESULTADOS_RAG,
    "Métricas de la línea RAG (con RAG)",
) == 1:
    print("\nTermina la calificación y presiona ENTER para continuar...")
    input()

Archivo: /home/moises/Documents/utb2026-2/ia/proyecto-final/chatutb/experiments/rag/respuestas_rag.csv
Filas: 50 | Calificadas: 50 | Sin calificar: 0
Errores de API: 0 | Sin respuesta generada: 0

MÉTRICAS PRINCIPALES
  Exactitud (%): 89.33
  Fidelidad (%): 86.66666666666667
  Tasa de alucinación (%): 2.0
  Respuesta perfecta (%): 78.0

OTRAS MÉTRICAS
  Preguntas totales: 50
  Preguntas calificadas: 50
  Cobertura de calificación (%): 100.0
  Errores de API: 0
  Sin respuesta generada: 0
  Exactitud plena (=3) (%): 80.0
  Exactitud aceptable (>=2) (%): 90.0
  Exactitud baja (<=1) (%): 10.0
  Fidelidad plena (%): 80.0
  Fidelidad baja (<=1) (%): 12.0
  Sin alucinación (%): 98.0
  Alucinación grave (=2) (%): 0.0

MÉTRICAS POR TIPO
                      tipo  n_evaluadas  %exactitud  %fidelidad  tasa_alucinacion  %respuesta_perfecta
       articulo_especifico           10   90.000000   90.000000              10.0            90.000000
     interpretacion_simple            8   79.166667   6